# 02 — Image Captioning en Google Colab

Notebook listo para ejecutar de arriba a abajo en Colab (GPU T4).
1. Clona el repo de GitHub.
2. Descarga Flickr8k de Kaggle.
3. Construye el vocabulario.
4. Entrena con W&B logging.
5. Genera captions de ejemplo.

**Antes de empezar:** *Runtime → Change runtime type → T4 GPU*

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repo

Cambia la URL por la de tu repo de GitHub Classroom.

In [ ]:
%cd /content
!git clone https://github.com/OVyla/Tweeter-Sentiment-Analysis-and-Classification.git proyectodeep
%cd proyectodeep

## 3. Instalar dependencias

In [ ]:
!pip install -q -r requirements.txt

## 4. Descargar Flickr8k de Kaggle

Necesitas tu `kaggle.json` (https://www.kaggle.com/settings → *Create New Token*).

In [ ]:
from google.colab import files
files.upload()  # selecciona kaggle.json

In [ ]:
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!mkdir -p data/flickr8k
!kaggle datasets download -d adityajn105/flickr8k -p data/flickr8k --unzip
!ls data/flickr8k | head

## 5. Construir vocabulario

In [ ]:
!python -m src.vocabulary --captions data/flickr8k/captions.txt --out data/flickr8k/vocab.pkl --threshold 5

## 6. Login en Weights & Biases

Te pedirá tu API key (https://wandb.ai/authorize).

In [ ]:
import wandb
wandb.login()

## 7. Entrenamiento

Con T4: ~5-7 min por época, 5 épocas ≈ 30 min.

In [ ]:
!python -m src.train \
    --epochs 5 \
    --batch-size 64 \
    --num-workers 2 \
    --lr 1e-3 \
    --backbone resnet50 \
    --wandb \
    --run-name baseline-resnet50-flickr8k

## 8. Generar captions de ejemplo

In [ ]:
import sys, random, glob
sys.path.insert(0, '/content/proyectodeep')

import torch
import matplotlib.pyplot as plt
from PIL import Image
from src.sample import load_checkpoint, caption_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = sorted(glob.glob('checkpoints/ckpt_epoch*.pt'))[-1]
encoder, decoder, vocab = load_checkpoint(ckpt, 'data/flickr8k/vocab.pkl', device)

samples = random.sample(glob.glob('data/flickr8k/Images/*.jpg'), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, path in zip(axes.flat, samples):
    cap = caption_image(path, encoder, decoder, vocab, device)
    ax.imshow(Image.open(path)); ax.axis('off')
    ax.set_title(cap, fontsize=10)
plt.tight_layout(); plt.show()

## 9. (Opcional) Subir captions de muestra a W&B

In [ ]:
import wandb
run = wandb.init(project='image-captioning', name='samples', job_type='inference')
table = wandb.Table(columns=['image', 'generated_caption'])
for path in samples:
    cap = caption_image(path, encoder, decoder, vocab, device)
    table.add_data(wandb.Image(path), cap)
run.log({'samples': table}); run.finish()